# Regresión Lineal: ¿Se puede predecir la popularidad de una canción?

Análisis de 5000+ tracks de Spotify para explorar si las características de audio
(danceability, energy, loudness…) explican la popularidad de una canción.

**Dataset:** [Spotify Tracks Dataset](https://www.kaggle.com/datasets/maharshipandya/-spotify-tracks-dataset)  
**Autor:** @aroaxinping  
**Fecha:** Abril 2026

---
## 0. Configuración del entorno

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

# Estilo
plt.rcParams.update({
    'figure.facecolor': '#0f0f0f',
    'axes.facecolor':   '#1a1a1a',
    'axes.edgecolor':   '#333',
    'text.color':       '#e0e0e0',
    'axes.labelcolor':  '#e0e0e0',
    'xtick.color':      '#999',
    'ytick.color':      '#999',
    'grid.color':       '#2a2a2a',
    'grid.linestyle':   '--',
    'font.family':      'monospace',
    'axes.titlecolor':  '#ffffff',
    'axes.titlesize':   13,
    'axes.titleweight': 'bold',
})

ACCENT  = '#e85d04'   # naranja
ACCENT2 = '#6a9ad4'   # azul
ACCENT3 = '#2dc653'   # verde

print('Entorno listo.')

---
## 1. Datos

| Dataset | Fuente | Registros |
|---|---|---|
| Spotify Tracks | [Kaggle](https://www.kaggle.com/datasets/maharshipandya/-spotify-tracks-dataset) | ~114k tracks (real) / 5k (sintético) |

> **Nota:** Si no tienes el CSV de Kaggle, el notebook genera un dataset sintético con distribuciones calibradas.

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path('../src').resolve()))

DATA_PATH = Path('../data/processed/spotify_clean.csv')

if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
    es_sintetico = False
    print(f'[OK] Datos cargados: {len(df)} tracks')
else:
    print('[INFO] Datos no encontrados. Generando dataset sintético...')
    print('[TIP]  Ejecuta: python src/fetch_spotify.py')
    from fetch_spotify import generate_synthetic_spotify
    df = generate_synthetic_spotify()
    es_sintetico = True

if es_sintetico:
    print('\n⚠️  AVISO: Datos sintéticos. Patrones realistas pero no son datos oficiales de Spotify.')

# Features numéricas que usaremos
FEATURES = [
    'danceability', 'energy', 'loudness', 'speechiness',
    'acousticness', 'instrumentalness', 'liveness', 'valence',
    'tempo', 'duration_ms', 'explicit',
]
TARGET = 'popularity'

print(f'\nColumnas: {df.columns.tolist()}')
print(f'Shape: {df.shape}')
df.head()

---
## 2. Exploración

> **Pregunta:** ¿Cómo se distribuye la popularidad y qué features tienen más relación con ella?

In [ ]:
# 2.1 Estadísticas descriptivas
print('--- Estadísticas del target ---')
print(df[TARGET].describe().round(2))
print(f'\nSkewness: {df[TARGET].skew():.3f}')
print(f'Kurtosis: {df[TARGET].kurtosis():.3f}')

In [ ]:
# 2.2 Distribución de popularidad
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(df[TARGET], bins=50, color=ACCENT, alpha=0.85, edgecolor='#333')
ax.axvline(df[TARGET].median(), color=ACCENT2, ls='--', lw=1.5, label=f'mediana = {df[TARGET].median():.0f}')
ax.axvline(df[TARGET].mean(), color=ACCENT3, ls='--', lw=1.5, label=f'media = {df[TARGET].mean():.1f}')
ax.set(xlabel='Popularity', ylabel='Frecuencia', title='Distribución de Popularity')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 2.3 Matriz de correlación
corr_cols = FEATURES + [TARGET]
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
    center=0, vmin=-1, vmax=1, ax=ax,
    annot_kws={'size': 8}, linewidths=0.5, linecolor='#333',
)
ax.set_title('Matriz de Correlación')
plt.tight_layout()
plt.show()

# Top correlaciones con popularity
print('\n--- Correlación con popularity ---')
target_corr = corr[TARGET].drop(TARGET).sort_values(key=abs, ascending=False)
for feat, val in target_corr.items():
    bar = '█' * int(abs(val) * 30)
    sign = '+' if val > 0 else '-'
    print(f'  {feat:20s} {sign}{abs(val):.3f}  {bar}')

In [ ]:
# 2.4 Scatter de las 4 features más correlacionadas
top4 = target_corr.head(4).index.tolist()

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, feat in zip(axes, top4):
    ax.scatter(df[feat], df[TARGET], alpha=0.15, s=5, color=ACCENT)
    # Línea de tendencia
    z = np.polyfit(df[feat], df[TARGET], 1)
    p = np.poly1d(z)
    x_line = np.linspace(df[feat].min(), df[feat].max(), 100)
    ax.plot(x_line, p(x_line), color=ACCENT2, lw=2)
    r = df[feat].corr(df[TARGET])
    ax.set(xlabel=feat, ylabel='popularity', title=f'{feat}\nr = {r:.3f}')
plt.tight_layout()
plt.show()

---
## 3. Regresión Lineal Simple

> **Pregunta:** ¿Cuánta varianza de la popularidad explica la feature más correlacionada por sí sola?

In [ ]:
# 3.1 Train/test split
best_feature = target_corr.index[0]
print(f'Feature más correlacionada: {best_feature} (r = {target_corr.iloc[0]:.3f})')

X_simple = df[[best_feature]]
y = df[TARGET]

X_train_s, X_test_s, y_train, y_test = train_test_split(
    X_simple, y, test_size=0.2, random_state=42
)
print(f'\nTrain: {len(X_train_s)} | Test: {len(X_test_s)}')

In [ ]:
# 3.2 Fit + métricas
lr_simple = LinearRegression()
lr_simple.fit(X_train_s, y_train)

y_pred_s = lr_simple.predict(X_test_s)

r2_simple = r2_score(y_test, y_pred_s)
rmse_simple = np.sqrt(mean_squared_error(y_test, y_pred_s))
mae_simple = mean_absolute_error(y_test, y_pred_s)

print(f'--- Regresión Lineal Simple ({best_feature}) ---')
print(f'  R²:   {r2_simple:.4f}')
print(f'  RMSE: {rmse_simple:.2f}')
print(f'  MAE:  {mae_simple:.2f}')
print(f'  Coef: {lr_simple.coef_[0]:.4f}')
print(f'  Intercept: {lr_simple.intercept_:.2f}')

In [ ]:
# 3.3 Visualización: predicción vs real
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter con línea de regresión
ax = axes[0]
ax.scatter(X_test_s, y_test, alpha=0.2, s=8, color=ACCENT, label='Real')
x_range = np.linspace(X_test_s.min().values[0], X_test_s.max().values[0], 100).reshape(-1, 1)
ax.plot(x_range, lr_simple.predict(x_range), color=ACCENT2, lw=2, label='Predicción')
ax.set(xlabel=best_feature, ylabel='Popularity', title=f'Regresión Simple — R² = {r2_simple:.3f}')
ax.legend()

# Predicted vs Actual
ax = axes[1]
ax.scatter(y_test, y_pred_s, alpha=0.2, s=8, color=ACCENT)
lims = [min(y_test.min(), y_pred_s.min()), max(y_test.max(), y_pred_s.max())]
ax.plot(lims, lims, '--', color=ACCENT3, lw=1.5, label='Predicción perfecta')
ax.set(xlabel='Popularity Real', ylabel='Popularity Predicha', title='Predicted vs Actual')
ax.legend()

plt.tight_layout()
plt.show()

---
## 4. Regresión Lineal Múltiple

> **Pregunta:** ¿Cuánto mejora el modelo si usamos todas las features de audio a la vez?

In [ ]:
# 4.1 Preparar datos con todas las features
X_multi = df[FEATURES].copy()
y = df[TARGET]

# Escalar features (importante cuando tienen rangos muy distintos)
scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X_multi),
    columns=FEATURES,
    index=X_multi.index,
)

X_train_m, X_test_m, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)
print(f'Train: {len(X_train_m)} | Test: {len(X_test_m)}')
print(f'Features: {FEATURES}')

In [ ]:
# 4.2 Fit + métricas
lr_multi = LinearRegression()
lr_multi.fit(X_train_m, y_train)

y_pred_m = lr_multi.predict(X_test_m)

r2_multi = r2_score(y_test, y_pred_m)
rmse_multi = np.sqrt(mean_squared_error(y_test, y_pred_m))
mae_multi = mean_absolute_error(y_test, y_pred_m)

print(f'--- Regresión Lineal Múltiple ({len(FEATURES)} features) ---')
print(f'  R²:   {r2_multi:.4f}')
print(f'  RMSE: {rmse_multi:.2f}')
print(f'  MAE:  {mae_multi:.2f}')

In [ ]:
# 4.3 Importancia de cada feature (coeficientes estandarizados)
coefs = pd.Series(lr_multi.coef_, index=FEATURES).sort_values(key=abs, ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
colors = [ACCENT if c > 0 else ACCENT2 for c in coefs]
ax.barh(coefs.index, coefs.values, color=colors, edgecolor='#333')
ax.axvline(0, color='#555', lw=0.8)
ax.set(xlabel='Coeficiente Estandarizado', title='Importancia de Features — Regresión Múltiple')

# Leyenda manual
from matplotlib.patches import Patch
ax.legend(
    handles=[Patch(color=ACCENT, label='↑ Más popularidad'), Patch(color=ACCENT2, label='↓ Menos popularidad')],
    loc='lower right',
)
plt.tight_layout()
plt.show()

---
## 5. Análisis de residuos

> **Pregunta:** ¿El modelo cumple los supuestos de la regresión lineal?

In [ ]:
# 5.1 Residuos del modelo múltiple
residuos = y_test.values - y_pred_m

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Distribución de residuos
ax = axes[0]
ax.hist(residuos, bins=40, color=ACCENT, alpha=0.85, edgecolor='#333')
ax.axvline(0, color=ACCENT3, ls='--', lw=1.5)
ax.set(xlabel='Residuo', ylabel='Frecuencia', title='Distribución de Residuos')

# Residuos vs predicción (homocedasticidad)
ax = axes[1]
ax.scatter(y_pred_m, residuos, alpha=0.15, s=8, color=ACCENT)
ax.axhline(0, color=ACCENT3, ls='--', lw=1.5)
ax.set(xlabel='Popularity Predicha', ylabel='Residuo', title='Residuos vs Predicción')

# Q-Q plot (normalidad)
ax = axes[2]
stats.probplot(residuos, dist='norm', plot=ax)
ax.get_lines()[0].set(color=ACCENT, markersize=3, alpha=0.5)
ax.get_lines()[1].set(color=ACCENT2, lw=2)
ax.set_title('Q-Q Plot')

plt.tight_layout()
plt.show()

# Test de normalidad
stat_sw, p_sw = stats.shapiro(residuos[:5000])  # Shapiro-Wilk max 5000
print(f'Shapiro-Wilk: stat={stat_sw:.4f}, p={p_sw:.2e}')
print(f'Media residuos: {residuos.mean():.4f}')
print(f'Std residuos:   {residuos.std():.2f}')

---
## 6. Comparación por género musical

> **Pregunta:** ¿El modelo funciona igual de bien para todos los géneros?

In [ ]:
# 6.1 R² por género (si hay columna genre)
if 'genre' in df.columns:
    # Usar los mismos índices de test
    test_idx = X_test_m.index
    df_test = df.loc[test_idx].copy()
    df_test['pred'] = y_pred_m
    df_test['residuo'] = residuos

    genre_metrics = []
    for genre, grp in df_test.groupby('genre'):
        if len(grp) >= 30:
            r2_g = r2_score(grp[TARGET], grp['pred'])
            rmse_g = np.sqrt(mean_squared_error(grp[TARGET], grp['pred']))
            genre_metrics.append({'genre': genre, 'n': len(grp), 'r2': r2_g, 'rmse': rmse_g})

    gm = pd.DataFrame(genre_metrics).sort_values('r2', ascending=False)

    fig, ax = plt.subplots(figsize=(10, 5))
    colors = [ACCENT if r > 0 else '#cc3333' for r in gm['r2']]
    ax.barh(gm['genre'], gm['r2'], color=colors, edgecolor='#333')
    ax.axvline(r2_multi, color=ACCENT2, ls='--', lw=1.5, label=f'R² global = {r2_multi:.3f}')
    ax.set(xlabel='R²', title='R² por Género Musical')
    ax.legend()
    plt.tight_layout()
    plt.show()

    print(gm.to_markdown(index=False))
else:
    print('Columna genre no disponible. Saltando análisis por género.')

---
## 7. Síntesis y conclusiones

| Modelo | R² | RMSE | MAE |
|---|---|---|---|
| Regresión Simple (1 feature) | — | — | — |
| Regresión Múltiple (11 features) | — | — | — |

*Los valores se calculan dinámicamente abajo.*

In [ ]:
# 7.1 Tabla resumen
resumen = pd.DataFrame([
    {
        'Modelo': f'Regresión Simple ({best_feature})',
        'R²': f'{r2_simple:.4f}',
        'RMSE': f'{rmse_simple:.2f}',
        'MAE': f'{mae_simple:.2f}',
    },
    {
        'Modelo': f'Regresión Múltiple ({len(FEATURES)} features)',
        'R²': f'{r2_multi:.4f}',
        'RMSE': f'{rmse_multi:.2f}',
        'MAE': f'{mae_multi:.2f}',
    },
])
print(resumen.to_markdown(index=False))

mejora_r2 = ((r2_multi - r2_simple) / max(abs(r2_simple), 0.001)) * 100
print(f'\nMejora en R² al añadir todas las features: {mejora_r2:+.1f}%')
print(f'Feature más importante: {coefs.index[-1]} (coef = {coefs.iloc[-1]:.3f})')
print(f'Feature menos importante: {coefs.index[0]} (coef = {coefs.iloc[0]:.3f})')

In [ ]:
# 7.2 Visualización final: simple vs múltiple
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, y_pred, title, r2 in [
    (axes[0], lr_simple.predict(X_test_s), f'Simple — R² = {r2_simple:.3f}', r2_simple),
    (axes[1], y_pred_m, f'Múltiple — R² = {r2_multi:.3f}', r2_multi),
]:
    ax.scatter(y_test, y_pred, alpha=0.15, s=8, color=ACCENT)
    lims = [0, 100]
    ax.plot(lims, lims, '--', color=ACCENT3, lw=1.5)
    ax.set(xlabel='Real', ylabel='Predicho', title=title, xlim=lims, ylim=lims)
    ax.set_aspect('equal')

plt.suptitle('Predicted vs Actual — Simple vs Múltiple', y=1.02, color='#fff', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n--- Conclusión ---')
print(f'La regresión lineal con {len(FEATURES)} features de audio explica un R² = {r2_multi:.3f}.')
print('Las características de audio son señales débiles para predecir popularidad.')
print('La popularidad depende más de factores externos (artista, marketing, playlists)')
print('que de las propiedades acústicas de la canción.')